In [ ]:
from scipy import interpolate, integrate
import matplotlib.pyplot as plt
import numpy as np
import numdifftools as nd

import darkprop as dp
from darkprop import units
from darkprop import constants

def f_Tr(Tr, dm, target):
    def dsigmadlogTr(logTr):
        Tr = np.exp(logTr)
        return dm.differentialCrossSection(Tr, target) * Tr
    norm = integrate.quad(dsigmadlogTr, np.log(1e-100), np.log(dm.maximumRecoilT(target)))[0]
    return np.array([dm.differentialCrossSection(T_r, target) / norm for T_r in np.array(Tr, ndmin=1)])

# Distribution of recoil energy and scattering angle

In [ ]:
sigma = 3e-28 * units.cm2
mchi = 0.01 * units.GeV
rn = dp.RandomNumber(1)

dm = dp.SIHelmDM(sigma, mchi)
target = dp.Target("O", 8, 16, 16*constants.mu)
earth = dp.PREMEarth()
for tar in earth.getTargets():
    dm.initCrossSectionCDF(1e-50, 1e10, tar, 10000)

T0 = 0.01
p0 = np.sqrt(T0*(T0 + 2*mchi))
dm.setP([0, 0, -p0])
dm.setR([0, 0, constants.rEarth - 1.4*units.km])
Tr = dm.scatteringSampleTr(target, rn)
dm.scatteringCosTheta(target, Tr)

In [ ]:
%%time
sigma = 3e-28 * units.cm2
mchi = 0.01 * units.GeV

rn = dp.RandomNumber(1)

dm = dp.SIHelmDM(sigma, mchi)
earth = dp.PREMEarth()
for tar in earth.getTargets():
    dm.initCrossSectionCDF(1e-50, 1e10, tar, 10000)

target = dp.Target("O", 8, 16, 16*constants.mu)

alpha = 0.5
nbins = 500

n = 1000000

fig, ax = plt.subplots()
fig2, ax2 = plt.subplots()

lss = ["--", "-.", ":"]
colors = ["tab:blue", "tab:orange", "tab:green"]

for i, T0 in enumerate([0.01, 0.1, 1.0]):
    p0 = np.sqrt(T0*(T0 + 2*mchi))
    dm.setP([0, 0, -p0])
    dm.setR([0, 0, constants.rEarth - 1.4*units.km])
    %time Tr_samples = np.array([dm.scatteringSampleTr(target, rn) for i in range(n)])
    costheta_samples = np.array([dm.scatteringCosTheta(target, Tr) for Tr in Tr_samples])

    Trmin = Tr_samples.min()

    Tr_plot = np.geomspace(Trmin, dm.maximumRecoilT(target) * 1.1, 1000)

    label = f"$T_{{\chi}} = {T0}$ GeV"
    ax.hist(Tr_samples, bins=np.geomspace(Tr_samples.min(), Tr_samples.max(), 201),
            density=True, alpha=alpha, color=colors[i])
    ax.plot(Tr_plot, f_Tr(Tr_plot, dm, target), ls=lss[i], color=colors[i], label=label)

    Trs = np.geomspace(Trmin, dm.maximumRecoilT(target), 100000)
    coss = np.array([dm.scatteringCosTheta(target, Tr) for Tr in Trs])

    ylog = interpolate.interp1d(coss, np.log(Trs), bounds_error=False, fill_value=-np.inf)
    def Tr_cos(x, inter=ylog):
        return np.exp(inter(x))

    def f_cos(cos, dm=dm, target=target, Tr_cos=Tr_cos):
        df_dx = nd.Derivative(Tr_cos)
        return f_Tr(Tr_cos(cos), dm, target) * np.abs(df_dx(cos))

    cos_plot = np.linspace(-1, 1, 2000)
    ax2.hist(costheta_samples, bins=np.linspace(costheta_samples.min(), costheta_samples.max(), 201),
             density=True, alpha=alpha, color=colors[i])
    ax2.plot(cos_plot, f_cos(cos_plot), color=colors[i], label=label, ls=lss[i])

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlim(1e-7, 1e-1)
ax.set_ylim(1e-4, 1e5)
ax.set_xlabel(r'$T_N$ [GeV]')
ax.set_ylabel(r'$p(T_N)$ [GeV$^{-1}$]')
text = r"$m_\chi = 10$ MeV" + "\n $N = $oxygen"
ax.text(0.8, 0.5, text, transform=ax.transAxes)
ax.legend()

ax2.set_xscale('linear')
ax2.set_yscale('linear')
ax2.set_xlim(-1, 1)
ax2.set_ylim(0, 3)
ax2.set_xlabel(r'$\cos\theta$')
ax2.set_ylabel(r'$p(\cos\theta)$')
ax2.text(0.05, 0.5, text, transform=ax2.transAxes)
ax2.legend()

# Distribution of free path

In [ ]:
%%time
sigma = 3e-28 * units.cm2
mchi = 0.01 * units.GeV

dm = dp.SIDM(sigma, mchi)

rn = dp.RandomNumber(2)
T0 = 0.1
# T0 = 0.5 * mchi * 0.001**2

n = 100000

earth = dp.HomoEarth()

mean_free_path = earth.meanFreePath(dm) / units.km

p0 = np.sqrt(T0*(T0 + 2*mchi))
dm.setP([0, 0, -p0])

r0 = [0, 0, constants.rEarth - 1.4*units.km]

free_path = []
for i in range(n):
    dm.setR(r0)
    earth.propagate(dm, rn)
    free_path.append(np.linalg.norm(np.array(dm.r.to_vec()) - np.array(r0)) / units.km)

In [ ]:
fig, ax = plt.subplots()
l = np.geomspace(np.min(free_path), np.max(free_path)*1.1, 100)
ax.plot(l, 1/mean_free_path * np.exp(-l/mean_free_path))
ax.hist(free_path, bins=100, density=True)
ax.set_yscale('log')
ax.set_xlabel(r'free path $l$ [km]')
ax.set_ylabel(r'PDF [km$^{-1}$]')
ax.set_title(f"$T_\mathrm{{i}}$ = {T0:.1f} GeV")
ax.text(0.6, 0.8, f"$\\lambda$ = {mean_free_path:.5f} km", transform=ax.transAxes)
print(f"mean free path from data: {np.mean(free_path)} km")